# 00 — MQA Methodology on Real Data

**Phase 1 deliverable · Commodity Hedging & Structuring Workbench**

This notebook replicates the **workflow** of the Citi MQA Forage Task 3 — cost-of-carry forward, Black-76 option pricing, Monte Carlo validation, scenario stress — entirely on **real frozen market data**. The simulation's example numbers are not used as inputs or answers; they described a different regime and only appear as labelled context in the closing comparison. Every number below is computed from the frozen coffee curve, the frozen SOFR series, and returns of the real data.

*Personal learning project extending the Citi MQA Forage simulation — not affiliated with or endorsed by Citi. Data: Yahoo Finance and FRED (unofficial/public sources, accepted and documented; gates fail closed to gold `GC=F` if coffee fails).*

## 1. Frozen data + gates

In [1]:
import sys
sys.path.insert(0, "../src")

import numpy as np
import pandas as pd

from hedging_workbench.data.frozen import FROZEN_DIR, latest_rate, load, verify
from hedging_workbench.pricing import black76
from hedging_workbench.sim import martingale_paths
from hedging_workbench.data.gates import evaluate
from hedging_workbench.data.universe import UNIVERSES

assert not verify("coffee"), "coffee manifest tampered"
assert not verify("rates"), "rates manifest tampered"
coffee = load(list(UNIVERSES["coffee"]))
gold = load(list(UNIVERSES["gold"]))
report = evaluate(coffee, gold)
print(report.summary())
print(f"risk-free anchor: SOFR = {latest_rate():.4f} (frozen from FRED)")
pd.DataFrame(report.series).T

[PASS] coffee: 9 series, 0 failures
risk-free anchor: SOFR = 0.0366 (frozen from FRED)


,rows,last_close,last_date
KC=F,674,324.25,2026-09-04
KCU26.NYB,675,324.25,2026-09-04
KCZ26.NYB,675,295.600006,2026-09-04
KCH27.NYB,613,287.399994,2026-09-04
KCK27.NYB,569,285.049988,2026-09-04
KCN27.NYB,529,283.399994,2026-09-04
KCU27.NYB,486,281.450012,2026-09-04
KCZ27.NYB,423,278.75,2026-09-04
KCH28.NYB,361,276.899994,2026-09-04


## 2. Real inputs — derived, not assumed

The simulation handed you S, r, d, σ on a slide. A desk derives them:

- **S** — the live front futures (`KC=F` continuous) as the spot proxy; coffee has no free spot index.
- **r** — frozen SOFR (overnight, annualised), the standard collateral discounting anchor.
- **d, y** — storage cost and convenience yield have no free series. The honest real-data move is to fold them into the **implied carry** read off the curve itself (`r + d - y` from calendar spreads) and disclose that decomposition is Phase 2 work.
- **σ** — no free coffee options data, so no implied vol. Preliminary estimate: **EWMA (RiskMetrics λ=0.94)** on real `KC=F` daily returns; GARCH calibration is Phase 2.

In [2]:
from hedging_workbench.data.universe import contract_label as label

S = coffee["KC=F"].iloc[-1] / 100          # spot proxy, $/lb
r = latest_rate()

# Implied annualised carry (r + d - y) from adjacent contract spreads
chain = [("KCU26.NYB", 0.00), ("KCZ26.NYB", 0.25), ("KCH27.NYB", 0.50),
         ("KCK27.NYB", 0.667), ("KCN27.NYB", 0.833), ("KCU27.NYB", 1.00),
         ("KCZ27.NYB", 1.25), ("KCH28.NYB", 1.50)]
rows = []
for (s1, t1), (s2, t2) in zip(chain, chain[1:]):
    f1, f2 = coffee[s1].iloc[-1] / 100, coffee[s2].iloc[-1] / 100
    rows.append({"from": label(s1), "to": label(s2),
                 "T (y)": round(t2 - t1, 3),
                 "F1": f1, "F2": f2,
                 "implied carry r+d-y": np.log(f2 / f1) / (t2 - t1)})
curve = pd.DataFrame(rows)
curve.style.format({"F1": "{:.4f}", "F2": "{:.4f}", "implied carry r+d-y": "{:+.1%}"})

,from,to,T (y),F1,F2,implied carry r+d-y
0,Sep 2026,Dec 2026,0.250000,3.2425,2.9560,-37.0%
1,Dec 2026,Mar 2027,0.250000,2.9560,2.8740,-11.3%
2,Mar 2027,May 2027,0.167000,2.8740,2.8505,-4.9%
3,May 2027,Jul 2027,0.166000,2.8505,2.8340,-3.5%
4,Jul 2027,Sep 2027,0.167000,2.8340,2.8145,-4.1%
5,Sep 2027,Dec 2027,0.250000,2.8145,2.7875,-3.9%
6,Dec 2027,Mar 2028,0.250000,2.7875,2.7690,-2.7%


### Reading the real curve

Every spread is **negative** — the market prices coffee in backwardation: the implied carry `r + d − y ≈ −8…−9%/yr` against SOFR at `+3.7%` means the convenience yield dominates. Holders of physical coffee are compensated for releasing it now; the sim's contango world (positive carry, `y ≈ 0`) describes the opposite scarcity regime.

## 3. Option pricing on the real forward — Black-76

In [3]:
# The sim priced a 6-month ATM option; replicate that structure on the real
# curve: underlying = Dec-26 futures (the liquid ~3-4 month tenor),
# strike = the futures itself (ATM-forward), T = time to mid-Dec expiry.
# One Black-76 implementation lives in the package (pricing.black76).
F = coffee["KCZ26.NYB"].iloc[-1] / 100
K = F                                   # ATM-forward
T = 0.28                                # ~3.4 months to ICE Dec-26 expiry

rets = np.log(coffee["KC=F"]).diff().dropna()
lam = 0.94
w = (1 - lam) * lam ** np.arange(len(rets) - 1, -1, -1)
ewma_var = np.sum(w * rets.values**2)
sigma = float(np.sqrt(ewma_var * 252))
print(f"real inputs: F = {F:.4f} $/lb, r = {r:.4f}, sigma(EWMA) = {sigma:.1%}, T = {T:.2f}y")

c_b76 = black76("call", F, K, T, sigma, r)
p_b76 = black76("put", F, K, T, sigma, r)
print(f"ATM-forward Black-76 call = {c_b76:.4f} $/lb  ({c_b76/F:.1%} of forward)")
print(f"ATM-forward Black-76 put  = {p_b76:.4f} $/lb  (put-call parity: "
      f"{abs(c_b76 - p_b76 - np.exp(-r*T)*(F-K)) < 1e-12})")

real inputs: F = 2.9560 $/lb, r = 0.0366, sigma(EWMA) = 46.2%, T = 0.28y
ATM-forward Black-76 call = 0.2844 $/lb  (9.6% of forward)
ATM-forward Black-76 put  = 0.2844 $/lb  (put-call parity: True)


## 4. Monte Carlo validation of the closed form

In [4]:
# Same check the MQA task teaches: does the simulated risk-neutral price
# agree with Black-76 on the REAL inputs? The shared Q-measure engine
# (sim.martingale_paths) walks the futures martingale — 10k GBM paths, seed 42.
FT = martingale_paths(F, sigma, T, steps=100, n_paths=10_000, seed=42)[:, -1]
payoffs = np.maximum(FT - K, 0)
c_mc = np.exp(-r * T) * payoffs.mean()
se = np.exp(-r * T) * payoffs.std() / np.sqrt(10_000)
print(f"MC   call = {c_mc:.4f} +/- {se:.4f} $/lb")
print(f"Black-76 = {c_b76:.4f} $/lb   within 1 SE: {abs(c_mc - c_b76) < se}")

MC   call = 0.2809 +/- 0.0048 $/lb
Black-76 = 0.2844 $/lb   within 1 SE: True


## 5. Scenario stress — the Task 3 sensitivity methodology, real base

In [5]:
# Frost: spot +12.5% AND vol spike; demand slump: -10%; vol risk premium: +5pts.
# Each scenario reprices the SAME Dec-26 ATM call off the real base.
scen = pd.DataFrame({
    "F ($/lb)":  [F, F*1.125, F*0.90, F, F],
    "sigma":     [sigma, min(sigma*1.6, 0.40), sigma, sigma+0.05, sigma*0.88],
}, index=["base (real)", "frost: +12.5% spot, vol spike", "demand slump: -10%",
          "vol risk premium: +5pts", "benign weather: vol -12%"])
scen["call ($/lb)"] = [black76("call", f, K, T, s, r) for f, s in zip(scen["F ($/lb)"], scen["sigma"])]
scen["delta vs base"] = scen["call ($/lb)"] / black76("call", F, K, T, sigma, r) - 1
scen.style.format({"F ($/lb)": "{:.4f}", "sigma": "{:.1%}",
                   "call ($/lb)": "{:.4f}", "delta vs base": "{:+.0%}"})

,F ($/lb),sigma,call ($/lb),delta vs base
base (real),2.9560,46.2%,0.2844,+0%
"frost: +12.5% spot, vol spike",3.3255,40.0%,0.4842,+70%
demand slump: -10%,2.6604,46.2%,0.1485,-48%
vol risk premium: +5pts,2.9560,51.2%,0.3150,+11%
benign weather: vol -12%,2.9560,40.6%,0.2504,-12%


## 6. Context vs the simulation's published example — different data, different regime

The MQA simulation's example answer priced a hypothetical contract off slide inputs ($1.20 spot, 25% vol, contango). This notebook priced the **same structure** off the frozen market:

| Dimension | Sim example (context only) | Real frozen curve (this notebook) |
|---|---|---|
| Price level | $1.20/lb | $3.24/lb front |
| Curve regime | contango, carry +3%/yr | **backwardation**, implied carry −8…−9%/yr |
| Convenience yield | assumed ≈ 0 | dominates the carry decomposition |
| Vol source | given 25% | EWMA on real returns (GARCH in Phase 2) |
| Rate | given 2% | frozen SOFR 3.66% |

**Why it matters for the next phases:** in backwardation a roaster's long futures hedge rolls *into* cheaper contracts (roll yield, not roll cost) — Task 4's collar strikes and Phase 3's hedge-month selection must be built on this curve, and the +12.5% frost scenario shows the same convexity the sim taught, on real numbers.

*The simulation's numbers are Citi-provided teaching material; they are quoted here only as labelled context for the regime difference. No Citi data or methodology is reproduced.*

## Reproducibility

- Frozen snapshots: `data/frozen/*.csv` + `manifest_coffee.json` / `manifest_gold.json` / `manifest_rates.json` (SHA-256 per file, verified above)
- Gates: `python -m hedging_workbench.data.gates` · frozen data: `python -m hedging_workbench.data.frozen [--universe coffee|gold] [--rates] [--verify]`
- Tests: `pytest` — 6 gate/manifest tests
- Limitations: unofficial Yahoo source; `KC=F` as spot proxy; EWMA vol is preliminary (Phase 2 GARCH); storage cost d not observable free — folded into implied carry, decomposition is Phase 2; frozen 2026-09-04 (curve) / 2026-09-05 (SOFR)